##### 소규모 모델 PEFT(LoRA)튜닝
- 가상의 문장을 생성
- train.jsonl, validation.jsonl 생성
- Transformers + PEFT(LoRA) fineTurning

In [14]:
model_name = 'Qwen/Qwen2.5-0.5B-Instruct'
train_file = r'finetune_data\train.jsonl'
valid_file = f'finetune_data\validation.jsonl'
output_dir = 'peft_output'
num_train_epochs = 3
per_device_train_batch_size = 4
learning_rate = 2e-4
lora_r = 8
lora_alpha = 16
lora_dropout = 0.05
use_4bit = False   # GPU + bitsandbytes  -> True
max_leng = 512

LABEL_DESC = {
    'DEF': '정의/목적/적용범위 조항',
    'RIGHT': '권리/의무/금지/책임 조항',
    'PROC': '신청/심사/조사/불복/처벌 절차 조항',
    'ORG': '기관/위원회/법원 등 조직의 설치/구성/권한 조항',
    'CRIT': '자격/요건/기준/기간/수치 조건 조항',
    'ETC': '시행일/경과조치/위임 등 기타 조항',
}
print('Config set:', model_name)

Config set: Qwen/Qwen2.5-0.5B-Instruct


In [15]:
# Causal lM : 기본학습 다음토큰을 예측
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

def build_inputs_and_labels(batch, tokenizer, max_length=512):
    '''prompt부분은 loss에서제외(-100) 정답 라벨 토큰만 학습'''
    inputs = []; labels = []
    for p, r in zip(batch['prompt'], batch['response']):
        full = p+' '+ r
        tokenized_full = tokenizer(full, truncation=True, max_length=max_length)
        tokenized_prompt = tokenizer(p, truncation=True, max_length=max_length)
        input_ids =  tokenized_full['input_ids']
        label_ids = input_ids.copy()

        prompt_len = len(tokenized_prompt['input_ids'])
        for i in range(min(prompt_len, len(label_ids))):
            label_ids[i]    = -100         
        inputs.append(input_ids)
        labels.append(label_ids)
    return {'input_ids' : inputs, 'labels':labels}

In [16]:
# 토크나이져, 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
load_kwargs = {'trust_remote_code':True}
if use_4bit:
    load_kwargs.update({'load_in_4bit':True,'device_map':'auto'})
model = AutoModelForCausalLM.from_pretrained(model_name, **load_kwargs)
if use_4bit:
    model = prepare_model_for_kbit_training(model)
print('model load')

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

model load


##### LoRA 구성 및 적용
- 모델내부 선형층 이름을 스캔해 LoRA를 걸 target_modules를 자동으로 찾기
- attention/MLP projection에 LoRA를 적용?
    - 출력헤드 전체를 모두 fine tune하면 비용증가
    - 출력에 영향이 큰 선형층을 효율적으로 조정
- LoRA는 기존 가중치 W를 직접 바꾸지 않고 행렬 업데이트를통해서 조금 수정
- 학습 파라메터가 줄어서 cpu나 저성능의 gpu에서 가능

In [17]:
import torch.nn as nn
def find_lora_target_modules(model):
    linear_leaf_names = set()
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            linear_leaf_names.add(name.split('.')[-1])

    preferred = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
    selected = [ m for m in preferred if m in linear_leaf_names]
    if not selected:
        fallback_keywords = ('proj','wq','wk','wv','wo','fc')
        selected = sorted([
            n for n in linear_leaf_names
            if any( k in  n for k in fallback_keywords) and n not in {'lm_head'}
        ])
    if not selected:
        raise ValueError(
            f'LoRA Target Module를 자동 탐색하지 못했습니다. 발견된 leaner leaf name : {linear_leaf_names}'
        )
    return selected, sorted(linear_leaf_names)

target_modules, linear_names = find_lora_target_modules(model)
print('Detected linear left names(samples) :',linear_names[:30])
print('Selected target_modules for LoRA', target_modules)

peft_config = LoraConfig(
    r=lora_r,
    lora_alpha=lora_alpha,
    target_modules=target_modules,
    lora_dropout=lora_dropout,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Detected linear left names(samples) : ['down_proj', 'gate_proj', 'k_proj', 'lm_head', 'o_proj', 'q_proj', 'up_proj', 'v_proj']
Selected target_modules for LoRA ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826
